In [52]:
import requests
import pandas as pd
import numpy as np
import re
from datetime import date

In [53]:
COLLECTION_DATE = date.today()

AWS Catalog Data

In [54]:
aws_offer_index_url = "https://pricing.us-east-1.amazonaws.com/offers/v1.0/aws/index.json"

response = requests.get(aws_offer_index_url)
print(response.status_code)

aws_offer_index = response.json()

200


In [55]:
aws_services = []

for service_code, info in aws_offer_index["offers"].items():
    aws_services.append({
        "provider": "AWS",
        "service_code": service_code,
        "service_name": info.get("offerCode"),
        "service_title": info.get("currentVersionUrl", "").split("/")[-3] if info.get("currentVersionUrl") else service_code,
        "source": "AWS Price List Offer Index",
        "collection_date": COLLECTION_DATE
    })

aws_service_df = pd.DataFrame(aws_services)

aws_service_df.head()

,provider,service_code,service_name,service_title,source,collection_date
0,AWS,comprehend,comprehend,comprehend,AWS Price List Offer Index,2026-06-04
1,AWS,AmazonMWAA,AmazonMWAA,AmazonMWAA,AWS Price List Offer Index,2026-06-04
2,AWS,ContactLensAmazonConnect,ContactLensAmazonConnect,ContactLensAmazonConnect,AWS Price List Offer Index,2026-06-04
3,AWS,AWSIoT,AWSIoT,AWSIoT,AWS Price List Offer Index,2026-06-04
4,AWS,VMwareCloudOnAWS,VMwareCloudOnAWS,VMwareCloudOnAWS,AWS Price List Offer Index,2026-06-04


In [56]:
aws_service_df.shape

(268, 6)

In [57]:
aws_service_df["service_name_clean"] = (
    aws_service_df["service_code"]
    .str.replace("Amazon", "", regex=False)
    .str.replace("AWS", "", regex=False)
    .str.replace("awskms", "KMS", regex=False)
)

aws_service_df.head()

,provider,service_code,service_name,service_title,source,collection_date,service_name_clean
0,AWS,comprehend,comprehend,comprehend,AWS Price List Offer Index,2026-06-04,comprehend
1,AWS,AmazonMWAA,AmazonMWAA,AmazonMWAA,AWS Price List Offer Index,2026-06-04,MWAA
2,AWS,ContactLensAmazonConnect,ContactLensAmazonConnect,ContactLensAmazonConnect,AWS Price List Offer Index,2026-06-04,ContactLensConnect
3,AWS,AWSIoT,AWSIoT,AWSIoT,AWS Price List Offer Index,2026-06-04,IoT
4,AWS,VMwareCloudOnAWS,VMwareCloudOnAWS,VMwareCloudOnAWS,AWS Price List Offer Index,2026-06-04,VMwareCloudOn


GCP Catalog Data

In [58]:
GCP_API_KEY = "YOUR_GCP_API_KEY"

In [59]:
def fetch_gcp_services(api_key):
    url = "https://cloudbilling.googleapis.com/v1/services"
    params = {"key": api_key}

    response = requests.get(url, params=params)

    print("Status:", response.status_code)
    if response.status_code != 200:
        print(response.text[:500])

    response.raise_for_status()

    return response.json().get("services", [])

In [60]:
gcp_services_raw = fetch_gcp_services(GCP_API_KEY)
len(gcp_services_raw)

Status: 200


1774

In [61]:
gcp_services = []

for item in gcp_services_raw:
    gcp_services.append({
        "provider": "GCP",
        "service_code": item.get("serviceId"),
        "service_name": item.get("displayName"),
        "service_title": item.get("displayName"),
        "source": "Google Cloud Billing Catalog API",
        "collection_date": COLLECTION_DATE
    })

gcp_service_df = pd.DataFrame(gcp_services)

gcp_service_df.head()

,provider,service_code,service_name,service_title,source,collection_date
0,GCP,0017-8C5E-5B91,OpenLogic CentOS 7.8 (v20200922) - Security Ha...,OpenLogic CentOS 7.8 (v20200922) - Security Ha...,Google Cloud Billing Catalog API,2026-06-04
1,GCP,002A-FAF7-9793,Single node file server,Single node file server,Google Cloud Billing Catalog API,2026-06-04
2,GCP,0042-8437-FF55,Adverity,Adverity,Google Cloud Billing Catalog API,2026-06-04
3,GCP,005A-0B51-C56E,Metabase Data Visualization & BI Platform,Metabase Data Visualization & BI Platform,Google Cloud Billing Catalog API,2026-06-04
4,GCP,0069-3716-5463,Qubole Open Data Lake Platform,Qubole Open Data Lake Platform,Google Cloud Billing Catalog API,2026-06-04


In [62]:
gcp_service_df.shape

(1774, 6)

Azure Catalog Data

In [63]:
azure_services = [
    # Compute
    ("Virtual Machines", "Compute"),
    ("Azure Kubernetes Service", "Compute"),
    ("Azure Functions", "Compute"),
    ("App Service", "Compute"),
    ("Azure Container Instances", "Compute"),
    ("Azure Batch", "Compute"),

    # Storage
    ("Blob Storage", "Storage"),
    ("Azure Files", "Storage"),
    ("Disk Storage", "Storage"),
    ("Queue Storage", "Storage"),
    ("Archive Storage", "Storage"),
    ("Data Lake Storage", "Storage"),

    # Database
    ("Azure SQL Database", "Database"),
    ("Azure Cosmos DB", "Database"),
    ("Azure Database for PostgreSQL", "Database"),
    ("Azure Database for MySQL", "Database"),
    ("Azure Cache for Redis", "Database"),

    # Analytics
    ("Azure Synapse Analytics", "Analytics"),
    ("Azure Databricks", "Analytics"),
    ("Azure Data Factory", "Analytics"),
    ("Microsoft Fabric", "Analytics"),
    ("Azure Stream Analytics", "Analytics"),

    # AI / ML
    ("Azure OpenAI Service", "AI / Machine Learning"),
    ("Azure Machine Learning", "AI / Machine Learning"),
    ("Azure AI Search", "AI / Machine Learning"),
    ("Azure AI Services", "AI / Machine Learning"),

    # Network
    ("Virtual Network", "Networking"),
    ("Load Balancer", "Networking"),
    ("Application Gateway", "Networking"),
    ("Azure CDN", "Networking"),
    ("Azure DNS", "Networking"),
    ("ExpressRoute", "Networking"),
    ("VPN Gateway", "Networking"),

    # Security
    ("Microsoft Defender for Cloud", "Security"),
    ("Key Vault", "Security"),
    ("Microsoft Sentinel", "Security"),
    ("Azure Firewall", "Security"),

    # Management
    ("Azure Monitor", "Management"),
    ("Azure Automation", "Management"),
    ("Azure Policy", "Management"),
    ("Azure Resource Manager", "Management")
]

azure_service_df = pd.DataFrame(
    azure_services,
    columns=["service_name", "category_manual"]
)

azure_service_df["provider"] = "Azure"
azure_service_df["service_code"] = azure_service_df["service_name"]
azure_service_df["service_title"] = azure_service_df["service_name"]
azure_service_df["source"] = "Curated Azure Service Catalog"
azure_service_df["collection_date"] = COLLECTION_DATE

azure_service_df.head()

,service_name,category_manual,provider,service_code,service_title,source,collection_date
0,Virtual Machines,Compute,Azure,Virtual Machines,Virtual Machines,Curated Azure Service Catalog,2026-06-04
1,Azure Kubernetes Service,Compute,Azure,Azure Kubernetes Service,Azure Kubernetes Service,Curated Azure Service Catalog,2026-06-04
2,Azure Functions,Compute,Azure,Azure Functions,Azure Functions,Curated Azure Service Catalog,2026-06-04
3,App Service,Compute,Azure,App Service,App Service,Curated Azure Service Catalog,2026-06-04
4,Azure Container Instances,Compute,Azure,Azure Container Instances,Azure Container Instances,Curated Azure Service Catalog,2026-06-04


In [64]:
azure_service_df.shape

(41, 7)

建立自動分類 function

In [65]:
def classify_service(service_name):
    text = str(service_name).lower()

    if any(k in text for k in [
        "compute", "ec2", "virtual machine", "vm", "lambda",
        "kubernetes", "container", "batch", "app engine",
        "cloud run", "functions", "elastic beanstalk"
    ]):
        return "Compute"

    elif any(k in text for k in [
        "storage", "s3", "blob", "file", "disk", "backup",
        "archive", "efs", "fsx", "filestore"
    ]):
        return "Storage"

    elif any(k in text for k in [
        "database", "rds", "sql", "aurora", "dynamodb",
        "cosmos", "spanner", "bigtable", "postgresql",
        "mysql", "redis", "memorystore", "documentdb"
    ]):
        return "Database"

    elif any(k in text for k in [
        "analytics", "bigquery", "dataflow", "dataproc",
        "athena", "emr", "glue", "redshift", "synapse",
        "databricks", "fabric", "stream"
    ]):
        return "Analytics"

    elif any(k in text for k in [
        "ai", "machine learning", "ml", "sagemaker",
        "vertex", "openai", "cognitive", "bedrock",
        "comprehend", "rekognition", "vision", "translate"
    ]):
        return "AI / Machine Learning"

    elif any(k in text for k in [
        "network", "vpc", "route", "dns", "load balancer",
        "cloudfront", "cdn", "gateway", "vpn", "interconnect",
        "expressroute"
    ]):
        return "Networking"

    elif any(k in text for k in [
        "security", "key", "kms", "iam", "identity",
        "defender", "firewall", "sentinel", "guardduty",
        "waf", "shield", "secrets"
    ]):
        return "Security"

    elif any(k in text for k in [
        "monitor", "logging", "cloudwatch", "operations",
        "management", "config", "policy", "automation"
    ]):
        return "Management"

    else:
        return "Other"

In [66]:
aws_service_df["category"] = aws_service_df["service_code"].apply(classify_service)
gcp_service_df["category"] = gcp_service_df["service_name"].apply(classify_service)

azure_service_df["category"] = azure_service_df["category_manual"]

In [67]:
standard_cols = [
    "provider",
    "service_code",
    "service_name",
    "service_title",
    "category",
    "source",
    "collection_date"
]

aws_final = aws_service_df[standard_cols].copy()
azure_final = azure_service_df[standard_cols].copy()
gcp_final = gcp_service_df[standard_cols].copy()

合併三家 service catalog

In [68]:
cloud_service_catalog_master = pd.concat(
    [aws_final, azure_final, gcp_final],
    ignore_index=True
)

cloud_service_catalog_master.head()

,provider,service_code,service_name,service_title,category,source,collection_date
0,AWS,comprehend,comprehend,comprehend,AI / Machine Learning,AWS Price List Offer Index,2026-06-04
1,AWS,AmazonMWAA,AmazonMWAA,AmazonMWAA,Other,AWS Price List Offer Index,2026-06-04
2,AWS,ContactLensAmazonConnect,ContactLensAmazonConnect,ContactLensAmazonConnect,Other,AWS Price List Offer Index,2026-06-04
3,AWS,AWSIoT,AWSIoT,AWSIoT,Other,AWS Price List Offer Index,2026-06-04
4,AWS,VMwareCloudOnAWS,VMwareCloudOnAWS,VMwareCloudOnAWS,Compute,AWS Price List Offer Index,2026-06-04


In [69]:
cloud_service_catalog_master.shape

(2083, 7)

In [70]:
cloud_service_catalog_master["provider"].value_counts()

provider
GCP      1774
AWS       268
Azure      41
Name: count, dtype: int64

In [71]:
cloud_service_catalog_master["category"].value_counts()

category
Other                    1436
Database                  137
AI / Machine Learning     119
Security                   88
Networking                 80
Analytics                  64
Storage                    61
Compute                    59
Management                 39
Name: count, dtype: int64

summary table

In [72]:
service_category_summary = (
    cloud_service_catalog_master
    .groupby(["provider", "category"])
    .agg(
        service_count=("service_name", "count")
    )
    .reset_index()
    .sort_values(["provider", "service_count"], ascending=[True, False])
)

service_category_summary

,provider,category,service_count
6,AWS,Other,198
0,AWS,AI / Machine Learning,17
1,AWS,Analytics,12
7,AWS,Security,10
5,AWS,Networking,8
8,AWS,Storage,8
2,AWS,Compute,5
3,AWS,Database,5
4,AWS,Management,5
14,Azure,Networking,7


In [73]:
pivot_summary = service_category_summary.pivot_table(
    index="category",
    columns="provider",
    values="service_count",
    fill_value=0
)

pivot_summary

provider,AWS,Azure,GCP
category,,,
AI / Machine Learning,17.0,4.0,98.0
Analytics,12.0,5.0,47.0
Compute,5.0,6.0,48.0
Database,5.0,5.0,127.0
Management,5.0,4.0,30.0
Networking,8.0,7.0,65.0
Other,198.0,0.0,1238.0
Security,10.0,4.0,74.0
Storage,8.0,6.0,47.0


In [74]:
cloud_service_catalog_master.to_csv(
    "cloud_service_catalog_master.csv",
    index=False
)

service_category_summary.to_csv(
    "service_category_summary.csv",
    index=False
)

pivot_summary.to_csv(
    "service_category_pivot_summary.csv"
)

print("Export completed.")

Export completed.


In [75]:
def classify_service_v3(name, description=""):

    text = f"{name} {description}".lower()

    if any(k in text for k in [
        "compute","vm","virtual machine","ec2","lambda",
        "container","kubernetes","eks","ecs",
        "cloud run","app engine","functions"
    ]):
        return "Compute"

    elif any(k in text for k in [
        "storage","bucket","blob","archive","backup",
        "disk","file storage","efs","fsx"
    ]):
        return "Storage"

    elif any(k in text for k in [
        "database","sql","nosql","mysql","postgres",
        "dynamodb","spanner","bigtable","redis"
    ]):
        return "Database"

    elif any(k in text for k in [
        "analytics","bigquery","athena","redshift",
        "spark","hadoop","etl","data warehouse",
        "stream","kinesis","dataflow","dataproc"
    ]):
        return "Analytics"

    elif any(k in text for k in [
        "ai","ml","machine learning","llm",
        "vision","speech","translate",
        "sagemaker","vertex ai","bedrock"
    ]):
        return "AI / Machine Learning"

    elif any(k in text for k in [
        "network","vpc","dns","cdn",
        "load balancer","gateway","vpn",
        "interconnect","pub/sub"
    ]):
        return "Networking"

    elif any(k in text for k in [
        "security","iam","identity","kms",
        "firewall","threat","sentinel",
        "defender","encryption"
    ]):
        return "Security"

    elif any(k in text for k in [
        "monitoring","logging","management",
        "operations","policy","resource manager",
        "governance","automation","observability"
    ]):
        return "Management"

    else:
        return "Other"

In [76]:
cloud_service_catalog_master["category_v3"] = (
    cloud_service_catalog_master.apply(
        lambda row: classify_service_v3(
            row["service_name"],
            row.get("description", "")
        ),
        axis=1
    )
)

In [77]:
cloud_service_catalog_master["category_v2"] = cloud_service_catalog_master["service_name"].apply(classify_service_v2)

cloud_service_catalog_master["category_v2"].value_counts()

category_v2
Other                    1450
Database                  140
AI / Machine Learning     123
Networking                 90
Security                   83
Analytics                  63
Storage                    62
Management                 40
Compute                    32
Name: count, dtype: int64

In [78]:
other_services = cloud_service_catalog_master[
    cloud_service_catalog_master["category_v3"] == "Other"
]["service_name"]

other_services.sample(50)

2011                                           OpenDocMan
1843                                                  H2O
1974                  Bitnami Pootle Certified by Bitnami
1079               Alfresco Community packaged by Bitnami
638                                                 Nginx
699                                             Roads API
193                              AWSElementalMediaConvert
1510                Bitnami TestLink Certified by Bitnami
2063                 NGINX, Inc NGINX Plus - Ubuntu 16.04
1662                                        Time Zone API
361                         PixStor Cloud - Balanced Tier
1210           Traffic Manager Standard Edition - 10 Mbps
1614     Visual Studio Pro 2015 on Windows Server 2012 R2
1133    TigerGraph Inc. TigerGraph Enterprise 3.0.6 (S...
1424                           Kissflow Digital Workplace
1828         Secure Your Cloud with Golden Hardened Image
1130                           WildFly Application Server
1412          

In [79]:
marketplace_keywords = [
    "bitnami",
    "ubuntu",
    "wordpress",
    "citrix",
    "linux",
    "windows server",
    "crm",
    "drupal",
    "grafana",
    "elasticsearch",
    "metabase",
    "nginx",
    "qualys",
    "bitcoin",
    "forum",
    "lamp",
    "redhat",
    "proxy",
    "scanner",
    "vm image",
    "marketplace",
    "certified by",
    "edition",
    "appliance",
    "server",
    "node"
]

In [80]:
def is_marketplace_service(service_name):

    text = str(service_name).lower()

    return any(
        keyword in text
        for keyword in marketplace_keywords
    )

In [81]:
cloud_service_catalog_clean = (
    cloud_service_catalog_master[
        ~cloud_service_catalog_master[
            "service_name"
        ].apply(is_marketplace_service)
    ]
    .copy()
)

In [82]:
cloud_service_catalog_clean["category"] = (
    cloud_service_catalog_clean["service_name"]
    .apply(classify_service_v3)
)

In [83]:
cloud_service_catalog_clean[
    "category"
].value_counts()

category
Other                    989
AI / Machine Learning    105
Database                  54
Networking                54
Analytics                 53
Security                  50
Compute                   49
Storage                   40
Management                33
Name: count, dtype: int64

In [84]:
def classify_service_v4(service_name):

    text = str(service_name).lower()

    # Compute
    if any(k in text for k in [
        "compute","vm","virtual machine","ec2","lambda",
        "container","kubernetes","eks","ecs",
        "cloud run","app engine","functions"
    ]):
        return "Compute"

    # Storage
    elif any(k in text for k in [
        "storage","bucket","blob","archive",
        "backup","disk","efs","fsx"
    ]):
        return "Storage"

    # Database
    elif any(k in text for k in [
        "database","sql","mysql","postgres",
        "dynamodb","spanner","bigtable",
        "redis","nosql"
    ]):
        return "Database"

    # Analytics
    elif any(k in text for k in [
        "analytics","bigquery","athena",
        "redshift","spark","etl",
        "data warehouse","dataflow",
        "dataproc","kinesis"
    ]):
        return "Analytics"

    # AI / ML
    elif any(k in text for k in [
        "ai","ml","machine learning",
        "vertex","bedrock","sagemaker",
        "vision","speech","translate",
        "openai"
    ]):
        return "AI / Machine Learning"

    # Networking
    elif any(k in text for k in [
        "network","vpc","dns","cdn",
        "gateway","vpn","load balancer",
        "interconnect"
    ]):
        return "Networking"

    # Security
    elif any(k in text for k in [
        "security","iam","identity",
        "kms","firewall","defender",
        "sentinel","guardduty"
    ]):
        return "Security"

    # Management
    elif any(k in text for k in [
        "monitor","logging","management",
        "policy","automation",
        "operations","resource manager"
    ]):
        return "Management"

    # DevOps
    elif any(k in text for k in [
        "devops","artifact","codebuild",
        "codepipeline","github",
        "gitlab","cloud build",
        "jenkins","ci/cd"
    ]):
        return "DevOps"

    # Messaging / Integration
    elif any(k in text for k in [
        "message","messaging","pub/sub",
        "event","eventbridge","sns",
        "sqs","queue","notification",
        "service bus"
    ]):
        return "Messaging / Integration"

    # Billing / Support
    elif any(k in text for k in [
        "billing","support",
        "enterprise support",
        "tax","subscription",
        "cost management"
    ]):
        return "Billing / Support"

    else:
        return "Other"

In [85]:
cloud_service_catalog_clean["category"] = (
    cloud_service_catalog_clean["service_name"]
    .apply(classify_service_v4)
)

In [86]:
cloud_service_catalog_clean["category"].value_counts()

category
Other                      967
AI / Machine Learning      103
Database                    54
Networking                  52
Security                    50
Compute                     49
Storage                     40
Management                  36
Analytics                   32
DevOps                      21
Billing / Support           14
Messaging / Integration      9
Name: count, dtype: int64

In [87]:
native_service_catalog = cloud_service_catalog_clean[
    cloud_service_catalog_clean["category"] != "Other"
].copy()

other_service_catalog = cloud_service_catalog_clean[
    cloud_service_catalog_clean["category"] == "Other"
].copy()

In [88]:
native_service_catalog.to_csv(
    "native_cloud_service_catalog.csv",
    index=False
)

other_service_catalog.to_csv(
    "other_marketplace_service_catalog.csv",
    index=False
)

service_category_summary = (
    native_service_catalog
    .groupby(["provider", "category"])
    .agg(service_count=("service_name", "count"))
    .reset_index()
)

service_category_summary.to_csv(
    "native_service_category_summary.csv",
    index=False
)